# **Máster en Behavioral Data Science**
## **Instituto de Formación Continua (IL3) - Universitat de Barcelona**
## **Módulo 9: Aprendizaje Profundo - Reto 1** - (Notebook 3/5)
Autores: **Meysam Madadi** & **Julio C. S. Jacques Junior**

---

# **Prerrequisitos**
- Consultar las instrucciones en los *Jupyter notebooks* anteriores.
- Ejecutar los *Jupyter notebooks* anteriores, en este caso:
 - *Jupyter notebook* 1.
 - *Jupyter notebook* 2.

# **Los objetivos de este Jupyter notebook**
- Practicar con **transfer learning** (aprendizaje por transferencia);
- Definir una **estrategia de entrenamiento**:
 - **Etapa 1:** Primero, entrenaremos nuestra "cabeza de regresión" (*regression head*). Para ello, "congelaremos" las capas de nuestro *backbone* (ResNet50) y entrenaremos solo las capas que hemos añadido al modelo. Esta etapa será más rápida que la siguiente, ya que solo se entrenan unas pocas capas.

 - **Etapa 2:** Luego, entrenaremos toda la red. Es decir, estableceremos todas las capas como "entrenables" para que los pesos de nuestro *backbone* también se puedan optimizar, junto con la "cabeza de regresión".
 - Finalmente, visualizamos la tendencia del entrenamiento de ambas etapas.
---

## Comprobando la versión de tensorflow

In [ ]:
# Este código fue probado en tensorflow 2.15.0
import tensorflow as tf
print(tf.__version__)

# Montando nuestro Google Drive para guardar/cargar nuestros resultados

In [ ]:
#--------------------------
MOUNT_GOOGLE_DRIVE = True
#--------------------------

if(MOUNT_GOOGLE_DRIVE==True):
  from google.colab import drive
  drive.mount('/content/gdrive')
  # Note, the default path will be: '/content/gdrive/MyDrive/'
  # In my case, the final path will be: '/content/gdrive/MyDrive/M09-P01/' as I
  # created a '/M09-P01/' folder in my google drive for this purpose.

# Cargando el modelo y los datos preprocesados desde Drive

In [ ]:
import numpy as np
import tensorflow as tf

with open('/content/gdrive/MyDrive/M09-P01/train.npy', 'rb') as f:
  X_train = np.load(f)
  Y_train = np.load(f)
  M_train = np.load(f)
with open('/content/gdrive/MyDrive/M09-P01/valid.npy', 'rb') as f:
  X_valid = np.load(f)
  Y_valid = np.load(f)
  M_valid = np.load(f)
with open('/content/gdrive/MyDrive/M09-P01/test.npy', 'rb') as f:
  X_test = np.load(f)
  Y_test = np.load(f)
  M_test = np.load(f)

model = tf.keras.models.load_model('/content/gdrive/MyDrive/M09-P01/init_model.h5')

# **Entrenando el modelo mediante un ajuste fino de la red pre-entrenada**
- Esta es una forma muy estándar de **transferencia de aprendizaje** en la que el conocimiento se transfiere desde una red que, en general, se entrenó en un gran conjunto de datos hacia otro modelo con la misma o una arquitectura similar. En nuestro caso, la red (ResNet50) se ha entrenado en más de 3 millones de datos faciales para la tarea de reconocimiento facial. La transferencia de conocimiento se realiza inicializando los pesos de la nueva red a partir de la red preentrenada.
- Como hemos visto anteriormente, hemos creado nuestra propia "cabeza de regresión" mediante la adición de algunas capas a nuestro *backbone*. En esta arquitectura, los pesos de la "cabeza de regresión" se inicializan con valores aleatorios.
- Una buena práctica para entrenar esta red es primero congelar los pesos de las capas preentrenadas y optimizar los pesos de la "cabeza de regresión" (de la capas añadidas, inicializados con valores aleatorios). Luego, entrenar (y refinar) toda la red. Para implementar esta estrategia, realizamos el entrenamiento en dos etapas (**Etapa 1** y **Etapa 2**).


## **Etapa 1:** Entrenando la "cabeza de regresión"
- Primero, **congelamos las primeras N=174 capas** de nuestra red para permitir el ajuste fino de las últimas capas. Para esto, definimos si una capa es entrenable o no.
- Luego, entrenamos la "cabeza de regresión" con un procedimiento similar al "entrenamiento desde cero". Es decir, el entrenamiento será similar, pero **las capas que estén congeladas no tendrán sus pesos actualizados**.


In [ ]:
import pickle
# freeze the first convolutional layers
counter = 0
for layer in model.layers:
  if counter <= 174:
    layer.trainable = False
  else:
    layer.trainable = True
  #print(counter, layer.name, layer.trainable)
  counter +=1

#### MODEL TRAINING ####
# defining the early stop criteria
es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=5)
# saving the best model based on val_loss
mc = tf.keras.callbacks.ModelCheckpoint('/content/gdrive/MyDrive/M09-P01/best_model_st1.h5', monitor='val_loss', mode='min', save_best_only=True)

# defining the optimizer
model.compile(tf.keras.optimizers.Adam(learning_rate=1e-4),loss=tf.keras.losses.MeanSquaredError(),metrics=['mae'])

# training the model
history = model.fit(X_train, Y_train, validation_data=(X_valid, Y_valid), batch_size=32, epochs=50, shuffle=True, verbose=1, callbacks=[es,mc])

# saving training history (for future visualization)
with open('/content/gdrive/MyDrive/M09-P01/train_history_st1.pkl', 'wb') as handle:
  pickle.dump(history.history, handle, protocol=pickle.HIGHEST_PROTOCOL)

## Evaluando el modelo entrenado (Etapa 1) en el conjunto de prueba

In [ ]:
# loading the (best) saved model
model = tf.keras.models.load_model('/content/gdrive/MyDrive/M09-P01/best_model_st1.h5')

# Evaluate the trained model on the test set
print('Evaluating on the test set')
predictions = model.predict(X_test, batch_size=32, verbose=1)

# Computing the Mean Absolute Error
# Also re-scaling the predictions to the range of "age" as the outputs are in the range of [0,1]
mae = np.mean(abs(predictions[:,0] - Y_test)*100)

# Next we print the average error. Note that the error is rescaled back to the range [0-100]
print('\nThe final mean absolute error (on the Test set)  is ' + str(mae) + ' years old.')

## Imprimiendo algunas predicciones generadas (Etapa 1)

In [ ]:
# printing some predictions and re-scaling the predicted values to the "age" range,
# using the normalization factor defined earlier, as output predictions are in
# the range of [0,1]
for i in range(0,10):
  print('predicted age = %.3f - Ground truth = %.3f' %(predictions[i]*100, Y_test[i]*100))

## **Etapa 2:** ajuste fino de toda la red
- A continuación, cargaremos el modelo entrenado en la Etapa 1, **estableceremos todas sus capas como "entrenables"** y **entrenaremos todo el modelo**.
- La red se entrena con una estrategia similar a la anterior. Sin embargo, la **tasa de aprendizaje se reduce** (de "1e-4" a "1e-5") para permitir que el algoritmo de entrenamiento intente aproximar mejor el error mínimo deseado.


In [ ]:
import pickle

# LOADING THE PREVIOUSLY TRAINED MODEL
model = tf.keras.models.load_model('/content/gdrive/MyDrive/M09-P01/best_model_st1.h5')

# setting all layers of the model to trainable
model.trainable = True

#### MODEL TRAINING ####
# defining the early stop criteria
es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=5)
# saving the best model based on val_loss
mc = tf.keras.callbacks.ModelCheckpoint('/content/gdrive/MyDrive/M09-P01/best_model_st2.h5', monitor='val_loss', mode='min', save_best_only=True)

# defining the optimizer
model.compile(tf.keras.optimizers.Adam(learning_rate=1e-5),loss=tf.keras.losses.MeanSquaredError(),metrics=['mae'])

# training the model
history = model.fit(X_train, Y_train, validation_data=(X_valid, Y_valid), batch_size=32, epochs=50, shuffle=True, verbose=1, callbacks=[es,mc])

# saving training history (for future visualization)
with open('/content/gdrive/MyDrive/M09-P01/train_history_st2.pkl', 'wb') as handle:
  pickle.dump(history.history, handle, protocol=pickle.HIGHEST_PROTOCOL)

## Evaluando el modelo entrenado (Etapa 2) en el conjunto de prueba

In [ ]:
# loading the (best) saved model
model = tf.keras.models.load_model('/content/gdrive/MyDrive/M09-P01/best_model_st2.h5')

# Evaluate the trained model on the test set
print('Evaluating on the test set')
predictions = model.predict(X_test, batch_size=32, verbose=1)

# Computing the Mean Absolute Error
# Also re-scaling the predictions to the range of "age" as the outputs are in the range of [0,1]
mae = np.mean(abs(predictions[:,0] - Y_test)*100)

# Next we print the average error. Note that the error is rescaled back to the range [0-100]
print('\nThe final mean absolute error (on the Test set)  is ' + str(mae) + ' years old.')

# Visualizando el historial de entrenamiento de ambas etapas (Etapa 1 y Etapa 2)
- Las curvas de ambas etapas están concatenadas.
- Se puede observar una caída de la función de pérdida (*Loss*) cerca del *epoch* 50, lo cual indica el final de la Etapa 1 de entrenamiento y el inicio de la Etapa 2.
- Tambien se puede observar como la Etapa 2 mejora el desempeño en comparación con la Etapa 1 y con respecto al error evaluado en el conjunto de validación.

In [ ]:
import pickle
from matplotlib import pyplot as plt

train_hist = pickle.load(open('/content/gdrive/MyDrive/M09-P01/train_history_st1.pkl',"rb"))
train_hist2 = pickle.load(open('/content/gdrive/MyDrive/M09-P01/train_history_st2.pkl',"rb"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
fig.suptitle('Training history (Stage 1 and Stage 2)', fontsize=14, fontweight='bold')

ax1.plot(train_hist['loss']+train_hist2['loss'])
ax1.plot(train_hist['val_loss']+train_hist2['val_loss'])
ax1.set(xlabel='epoch', ylabel='Loss')
ax1.legend(['train', 'valid'], loc='upper right')

ax2.plot(train_hist['mae']+train_hist2['mae'])
ax2.plot(train_hist['val_mae']+train_hist2['val_mae'])
ax2.set(xlabel='epoch', ylabel='MAE')
ax2.legend(['train', 'valid'], loc='upper right')